## 🏗️ LeetCode 703: Kth Largest Element in a Stream
---

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #e3f2fd; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> A min-heap of size k maintains the k largest elements seen so far. The root — the smallest in the heap — is always the kth largest overall. When a new element arrives: push it, and if the heap exceeds size k, pop the root (evicting the smallest of the top k, which is no longer in the top k).
</div>

### 🛠️ The Mathematical Model

Maintain the invariant: heap contains exactly the k largest elements. The root is the minimum of the top k = the kth largest.

$$\text{heap root} = \min(\text{top } k \text{ elements}) = k\text{th largest element}$$

---

### 📋 Problem

Design a class that finds the kth largest element in a stream. Initialize with integer `k` and array `nums`. Implement `add(val)` which appends a value to the stream and returns the kth largest element.

**Example 1:**
```
Input:  k = 3, nums = [4, 5, 8, 2]
add(3) → 4 | add(5) → 5 | add(10) → 5 | add(9) → 8 | add(4) → 8
```

**Constraints:** 1 ≤ k ≤ 10⁴ | 0 ≤ nums.length ≤ 10⁴ | -10⁴ ≤ nums[i] ≤ 10⁴ | At most 10⁴ calls to add | It's guaranteed kth largest always exists

### 🧠 Choose Your Mental Model

<table style="width:100%; border-collapse: collapse;">
    <tr style="background-color: #f2f2f2; text-align: left;">
        <th style="padding: 12px; border: 1px solid #ddd;">Model</th>
        <th style="padding: 12px; border: 1px solid #ddd;">The "Story"</th>
        <th style="padding: 12px; border: 1px solid #ddd;">Mechanism</th>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Bouncer's VIP List</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"Maintain a VIP list of the top k people. The person at the door of the list (the least VIP) is position k. When someone new arrives, check if they belong. If yes, add them and evict whoever is now position k+1."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">min-heap of size k — root = k-th VIP. heappush + heappop when size exceeds k</td>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Sliding Cutoff</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"The kth largest is a moving bar. New values above the bar replace the bar. Values below the bar are irrelevant. The min-heap's root IS the bar."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Heap root tracks the current cutoff. Elements below it are outside the top k.</td>
    </tr>
</table>

---

### ⚠️ Performance Warning

<div style="padding: 10px; border: 1px solid #ffe58f; background-color: #fffbe6; border-radius: 4px;">
    <strong>Note:</strong> Sorting the entire list after each add() is O(n log n) per call. With 10,000 add() calls, total work is O(n² log n). Min-heap of size k makes each add() O(log k) — far better.
</div>

## 🐢 Approach 1: Brute Force — $O(n \log n)$ per add

In [ ]:
class KthLargest_Brute:
    """
    Brute Force: Sort entire list after each add()
    Time: O(n log n) per add | Space: O(n)
    """
    def __init__(self, k, nums):
        self.k = k
        self.nums = nums

    def add(self, val):
        self.nums.append(val)
        self.nums.sort(reverse=True)
        return self.nums[self.k - 1]


kl = KthLargest_Brute(3, [4, 5, 8, 2])
print(kl.add(3))    # Expected: 4
print(kl.add(5))    # Expected: 5
print(kl.add(10))   # Expected: 5

## 🔬 Complexity Analysis: $O(n \log n)$ vs. $O(\log k)$ per add

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> We don't need to know all n elements in sorted order — we only need the k largest. A min-heap of fixed size k maintains exactly this. Push: O(log k). Pop: O(log k). The heap size never grows beyond k, so each add() is O(log k) regardless of how many total elements have been seen.
</div>

---

## 📉 Why Brute Force Fails: The $O(n \log n)$ Trap

Sort + index after each add() re-sorts the growing list. After m add() calls on initial n elements, total work is O((n+m) log(n+m)) per call.

| Input Type | Brute Force Performance | Reason |
|------------|------------------------|--------|
| **10,000 add() calls** | O(n² log n) total | Each add re-sorts the full list |
| **Streaming data** | Unbounded cost | Sort cost grows with stream length |

---

## 🚀 The Optimal Approach: $O(\log k)$ per add

Keep a min-heap of exactly k elements. Push the new value, then pop if size > k. The root is always the kth largest.

### The Key Lifecycle Rule
1. **Push new element** onto the heap — O(log k)
2. **If heap size > k:** pop the root (smallest element, no longer in top k) — O(log k)
3. **Root is the answer** — O(1)

---

## ✅ Mathematical Proof

The heap invariant: heap contains the k largest elements seen. When we push a new element:
- If new > root: new displaces root in top k, root is evicted
- If new ≤ root: new is below the kth largest, immediately evicted
After each add, heap contains the k largest, root = kth largest. ✓

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> By fixing the heap size at k, we get O(log k) operations — independent of total stream length. The heap size is a constant; only the heap structure changes.
</div>

## 🚀 Approach 2: Min-Heap of Size k — $O(\log k)$ per add
---

Instead of sorting on each add, we **maintain a fixed-size min-heap** where the root is always the kth largest.

As we iterate:
1. heappush(heap, val) — add the new element
2. If len(heap) > k: heappop(heap) — evict the smallest (it's no longer top k)
3. heap[0] is the kth largest

In [ ]:
import heapq

class KthLargest:
    """
    Optimal: Min-Heap of size k
    __init__: O(n log k) | add: O(log k) | Space: O(k)
    """
    def __init__(self, k: int, nums: list[int]):
        self.k = k
        self.heap = []
        for num in nums:
            self.add(num)   # reuse add() logic to build heap correctly

    def add(self, val: int) -> int:
        heapq.heappush(self.heap, val)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)   # evict the smallest of the top k+1
        return self.heap[0]            # root = kth largest


kl = KthLargest(3, [4, 5, 8, 2])
print("Optimal:", kl.add(3))    # Expected: 4
print("Optimal:", kl.add(5))    # Expected: 5
print("Optimal:", kl.add(10))   # Expected: 5
print("Optimal:", kl.add(9))    # Expected: 8
print("Optimal:", kl.add(4))    # Expected: 8

## 🎤 5 Interview Q&A

### Q1: Why a min-heap instead of a max-heap for "kth largest"?

**Answer:** The root of a min-heap of size k is the smallest of the k largest elements — which is the kth largest overall. A max-heap would give us the largest element as root, which isn't directly the kth largest without knowing k's position. The min-heap of size k IS the top-k filter: everything below root isn't in the top k.

---

### Q2: What if k equals 1 — what does the heap become?

**Answer:** A heap of size 1 — always holds the single maximum element. `add()` returns the running maximum of the entire stream. The heap simply tracks the one largest value seen.

---

### Q3: What is the time complexity of the optimal approach and why?

**Answer:** `__init__`: O(n log k) — n elements, each heappush is O(log k). `add()`: O(log k) — one heappush + at most one heappop, each O(log k). Space: O(k) — the heap never exceeds k elements. If k << n, this is a massive improvement over O(n log n) sorting.

---

### Q4: How does this pattern apply to real-time leaderboards or percentile tracking?

**Answer:** Exact same structure. For a live P95 (95th percentile) response time across n = 1,000,000 requests: a min-heap of size k = 50,000 (top 5%) tracks the P95 boundary as a stream. Each new response time is pushed; if it exceeds the current P95 threshold (heap root), it enters the top 5% and evicts the old boundary. O(log 50000) ≈ 16 operations per request.

---

### Q5: What are the edge cases to watch for?

**Answer:** (1) Initial nums array shorter than k — heap has fewer than k elements, add() still returns heap[0] correctly once heap size reaches k. (2) Add value smaller than all existing — heappush then immediately heappop (size > k case), root unchanged. (3) k = 1 — heap always size 1, returns running maximum. (4) All values equal — heap fills with equal values, root = that value, always returned.

## 📚 Key Terminology

| Term | Definition |
|------|------------|
| **Min-Heap** | A complete binary tree where each node is ≤ its children — the root is always the minimum element |
| **Max-Heap** | A complete binary tree where each node is ≥ its children — the root is always the maximum |
| **heapq** | Python's standard library heap module — implements a min-heap on a list |
| **heappush** | Add an element to the heap in O(log n) — maintains heap property |
| **heappop** | Remove and return the minimum element in O(log n) — maintains heap property |
| **Top-K Problem** | Finding the k largest (or smallest) elements from n elements — heap approach: O(n log k) |
| **Heap Invariant** | At all times, heap[0] is the minimum (min-heap) or maximum (max-heap) element |
| **Stream** | An unbounded sequence of values arriving over time — the algorithm must handle each value as it arrives |

## 💼 The Citi Narrative

**Context:** Citi monitored 6,000 server endpoints for CPU utilization. Business requirement: maintain a real-time list of the Top 100 most-utilized servers at any point — updated every minute as new readings arrived.

**Scenario:** 6,000 new CPU readings arrive every minute. Sorting all 6,000 to find the top 100 takes O(6000 log 6000) ≈ 72,000 operations per minute. A min-heap of size 100 needs O(6000 × log 100) ≈ 40,000 operations — and the heap can be maintained incrementally without re-sorting from scratch.

**How this pattern applied:** KthLargest(k=100, nums=initial_readings) built the initial heap once. Each subsequent minute, 6,000 `add()` calls updated the heap — only elements entering the top 100 caused structural changes. The heap root at any point was the 100th highest CPU reading — the alerting threshold boundary.

**Impact:** Real-time top-100 server list maintained with O(n log k) cost instead of O(n log n) sort per minute. The heap root served as a dynamic alerting threshold — servers that entered the top 100 triggered investigation automatically, without a fixed static threshold that didn't adapt to fleet-wide load patterns.

In [ ]:
# -------------------------------------------------------
# PRACTICE: Find the Kth SMALLEST element in a stream.
# Hint: use a MAX-heap of size k (negate values for max-heap in Python).
# The root of a max-heap of size k = the kth smallest.
# -------------------------------------------------------

import heapq

class KthSmallest:
    def __init__(self, k: int, nums: list[int]):
        self.k = k
        self.heap = []   # max-heap (negate values)
        for num in nums:
            self.add(num)

    def add(self, val: int) -> int:
        # Your solution here
        pass


# Test
ks = KthSmallest(3, [4, 5, 8, 2])
print(ks.add(3))    # Expected: 4
print(ks.add(1))    # Expected: 3

## 🎯 Summary: Key Takeaways

### The Pattern
**Min-Heap of Size k** — Track the k largest elements in a stream; the root is always the kth largest

### When to Use It
- ✅ Streaming data, need running kth largest/smallest
- ✅ "Top K" without sorting all n elements
- ✅ Real-time leaderboards, percentile tracking, capacity monitoring
- ❌ **Don't use when:** You need all k elements sorted (heap gives unordered top k) — need heapq.nlargest()

### Complexity
| Approach | Time (per add) | Space |
|----------|---------------|-------|
| Brute Force (sort) | $O(n \log n)$ | $O(n)$ |
| Optimal (Min-Heap size k) | $O(\log k)$ | $O(k)$ |

### Interview Confidence Checklist
- [ ] Can explain why min-heap of size k gives the kth largest
- [ ] Can write the add() method from memory
- [ ] Can explain why we evict when size > k
- [ ] Can apply to percentile tracking use case with specific numbers
- [ ] Can describe the k=1 edge case

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master Kth Largest in a Stream and you've unlocked the pattern behind real-time leaderboards, SLO percentile monitoring, and capacity alert thresholds. 🚀